In [3]:
! pip install transformers
! pip install datasets
! pip install sentencepiece

  Using cached pyarrow-19.0.1-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (3.3 kB)
  Using cached dill-0.3.8-py3-none-any.whl.metadata (10 kB)
  Using cached aiohttp-3.11.13-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.7 kB)
  Using cached aiohappyeyeballs-2.4.6-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.3.2-py2.py3-none-any.whl.metadata (3.8 kB)
  Using cached attrs-25.1.0-py3-none-any.whl.metadata (10 kB)
  Using cached frozenlist-1.5.0-cp312-cp312-manylinux_2_5_x86_64.manylinux1_x86_64.manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (13 kB)
  Using cached multidict-6.1.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.0 kB)
  Using cached propcache-0.3.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (10 kB)
  Using cached yarl-1.18.3-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (69 kB)
Using cached dill-0.3.8-py3-none-any.whl (116 kB)
Using cached aiohttp-3.11.

In [1]:
import pandas as pd
import re
from datasets import Dataset
import time
import torch
import pandas as pd
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from transformers import DataCollatorForSeq2Seq


/home/pavan/Ds/icondf/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [17]:
df = pd.read_csv('/content/drive/MyDrive/dataset/combined_dataset.csv')
print(df.head())


                                            question  \
0                     What is (are) Menkes Disease ?   
1       What are the treatments for Menkes Disease ?   
2           What is the outlook for Menkes Disease ?   
3  what research (or clinical trials) is being do...   
4       What is (are) Pelizaeus-Merzbacher Disease ?   

                                              answer        qtype  \
0  Menkes disease is caused by a defective gene n...  information   
1  Treatment with daily copper injections may imp...    treatment   
2  Since newborn screening for this disorder is n...      outlook   
3  Recent research sponsored by the NINDS develop...     research   
4  Pelizaeus-Merzbacher disease (PMD) is a rare, ...  information   

                          focus synonyms semantic_group  
0                Menkes Disease      NaN      Disorders  
1                Menkes Disease      NaN      Disorders  
2                Menkes Disease      NaN      Disorders  
3               

In [18]:
df.isnull().sum()

# Optional For Now : Drop rows where either the question or answer is missing
#df = df.dropna(subset=['question', 'answer'])

,0
question,0
answer,0
qtype,0
focus,14
synonyms,5512
semantic_group,0


This prompt formatting is a common practice when adapting QA datasets to T5-style models, including domain‑specific ones like BioT5. The prefix helps the model understand the task, while the clear separation between input (question) and output (answer) aligns with the text‑to‑text training objective.

In [19]:
# For a medical QA task, you might include a domain-specific prefix:
df['input_text'] = "medical question: " + df['question'].str.strip() + " answer:"
df['target_text'] = df['answer'].str.strip()


In [20]:
# Optionally, apply additional normalization (e.g., lowercasing) if your tokenizer expects that:
def normalize(text):
    return " ".join(text.lower().split())

df['input_text'] = df['input_text'].apply(normalize)
df['target_text'] = df['target_text'].apply(normalize)

In [21]:
df[['input_text', 'target_text']].sample()

,input_text,target_text
13774,medical question: how to diagnose chronic myel...,tests that examine the blood and bone marrow a...


In [22]:
# Display a few rows to verify the new columns
print(df[['input_text', 'target_text']].head())

                                          input_text  \
0  medical question: what is (are) menkes disease...   
1  medical question: what are the treatments for ...   
2  medical question: what is the outlook for menk...   
3  medical question: what research (or clinical t...   
4  medical question: what is (are) pelizaeus-merz...   

                                         target_text  
0  menkes disease is caused by a defective gene n...  
1  treatment with daily copper injections may imp...  
2  since newborn screening for this disorder is n...  
3  recent research sponsored by the ninds develop...  
4  pelizaeus-merzbacher disease (pmd) is a rare, ...  


In [26]:
# Convert the Pandas DataFrame to a Hugging Face Dataset

dataset = Dataset.from_pandas(df[['input_text', 'target_text']])

In [29]:
# Define your model name. For our example, we use FLAN-T5-base.
model_name = 'google/flan-t5-base'



In [30]:
# Load the base model with bfloat16 precision (if your hardware supports it)
original_model = AutoModelForSeq2SeqLM.from_pretrained(model_name, torch_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

In [32]:
def print_number_of_trainable_model_parameters(model):
    trainable_model_params = sum(param.numel() for _, param in model.named_parameters() if param.requires_grad)
    all_model_params = sum(param.numel() for _, param in model.named_parameters())
    return (
        f"trainable model parameters: {trainable_model_params}\n"
        f"all model parameters: {all_model_params}\n"
        f"percentage of trainable model parameters: {100 * trainable_model_params / all_model_params:.2f}%"
    )

print(print_number_of_trainable_model_parameters(original_model))

trainable model parameters: 247577856
all model parameters: 247577856
percentage of trainable model parameters: 100.00%


In [33]:
# Define a tokenization function tailored to your task.
def tokenize_function(examples):
    model_inputs = tokenizer(
        examples['input_text'],
        max_length=512,
        truncation=True,
        padding="max_length"  # Ensure fixed length
    )
    # Tokenize target text similarly
    labels = tokenizer(
        text_target=examples['target_text'],
        max_length=128,
        truncation=True,
        padding="max_length"
    )
    model_inputs['labels'] = labels['input_ids']
    return model_inputs



In [34]:
# Apply tokenization to your dataset
# tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/16402 [00:00<?, ? examples/s]

In [36]:
# Define output directory
output_dir = '/content/drive/MyDrive/dataset/results/peft-medical-qa-training-on-T5'

In [41]:
# Configure LoRA via PEFT: for a medical QA task we want to adapt the query ("q") and value ("v") projections.
lora_config = LoraConfig(
    r=128,                   # Rank
    lora_alpha=32,
    target_modules=["q", "v"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM  # This indicates a sequence-to-sequence language modeling task.
)

# Wrap the model with LoRA adapters
peft_model = get_peft_model(original_model, lora_config)
print(print_number_of_trainable_model_parameters(peft_model))

trainable model parameters: 14155776
all model parameters: 261733632
percentage of trainable model parameters: 5.41%


In [47]:
# Apply tokenization to your dataset
tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Alternatively, use a dynamic padding collator:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=peft_model, padding=True)


Map:   0%|          | 0/16402 [00:00<?, ? examples/s]

In [48]:
# Split dataset into training and (optional) validation sets.
# Here we use an 80/20 split:
split_dataset = tokenized_dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = split_dataset['train']
eval_dataset = split_dataset['test']


In [52]:
# Set up training arguments (adjust num_train_epochs and max_steps as needed)
peft_training_args = TrainingArguments(
    output_dir=output_dir,
    auto_find_batch_size=True,
    learning_rate=1e-3,      # Typically higher for PEFT
    num_train_epochs=3,      # Increase this for more training
    logging_steps=50,
    save_steps=500,
    eval_strategy="steps",
    eval_steps=500,
    max_steps=3501           # Set this to a proper value for full training
)


In [53]:
# Initialize the Trainer with your train dataset (and optionally eval_dataset)
peft_trainer = Trainer(
    model=peft_model,
    args=peft_training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator  # Use dynamic padding
)

In [54]:
# Start training
peft_trainer.train()

Step,Training Loss,Validation Loss
500,1.979400,1.764802
1000,1.986600,1.726187
1500,1.931900,1.697777
2000,1.866900,1.678754
2500,1.854400,1.662904
3000,1.855600,1.648137
3500,1.759600,1.643944


TrainOutput(global_step=3501, training_loss=1.8795779330905455, metrics={'train_runtime': 1708.7132, 'train_samples_per_second': 16.391, 'train_steps_per_second': 2.049, 'total_flos': 1170484225873920.0, 'train_loss': 1.8795779330905455, 'epoch': 2.13345521023766})

In [55]:
# Save the fine-tuned model and tokenizer
peft_model_path = "/content/drive/MyDrive/dataset/results/t5-peft_model"
peft_trainer.model.save_pretrained(peft_model_path)
tokenizer.save_pretrained(peft_model_path)

('/content/drive/MyDrive/dataset/results/t5-peft_model/tokenizer_config.json',
 '/content/drive/MyDrive/dataset/results/t5-peft_model/special_tokens_map.json',
 '/content/drive/MyDrive/dataset/results/t5-peft_model/spiece.model',
 '/content/drive/MyDrive/dataset/results/t5-peft_model/added_tokens.json',
 '/content/drive/MyDrive/dataset/results/t5-peft_model/tokenizer.json')

In [2]:
# Load the fine-tuned model and tokenizer
peft_model_path = "/home/pavan/Ds/icondf/results-20250302T115210Z-001/results/t5-peft_model"
model = AutoModelForSeq2SeqLM.from_pretrained(peft_model_path, torch_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(peft_model_path)


In [3]:
# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): lora.Linear(
                (base_layer): Linear(in_features=768, out_features=768, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=768, out_features=128, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=128, out_features=768, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k): Linear(in_fe

In [4]:
# Define a test query using the same prompt style used during training.
# For example, if you prepended "medical question:" and appended " answer:" during training:
test_query = "medical question: what is diabetes? answer:"

# Tokenize the input
input_ids = tokenizer.encode(test_query, return_tensors="pt").to(device)


In [17]:
# # Generate a response from the model
# # You can adjust parameters like max_length and num_beams as needed.
# outputs = model.generate(input_ids, max_length=128, num_beams=5, early_stopping=True)

# Generate the response with adjusted parameters to reduce repetition.
# Here, we use beam search with a repetition penalty and a no_repeat_ngram_size.
outputs = model.generate(
    input_ids,
    max_length=128,
    num_beams=5,
    repetition_penalty=2.5,
    no_repeat_ngram_size=3,
    early_stopping=True
)
# Decode the generated tokens back to a string
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Response:", response)

Response: glucose levels are the amount of sugar in your blood. they vary depending on how much glucose you eat and how often you drink it. some people don't need to worry about their glucose levels, while others do not. for example, diabetics may have more glucose than non-diabetes.


In [16]:
# Define a test query using the same prompt style used during training.
# For example, if you prepended "medical question:" and appended " answer:" during training:
test_query = "medical question: what is glucose levels? answer:"

# Tokenize the input
input_ids = tokenizer.encode(test_query, return_tensors="pt").to(device)


In [18]:
import gradio as gr
print(gr.__version__)


5.20.0
